In [21]:
import pandas as pd

path = "filtered.csv"
df = pd.read_csv(path)

df["ModelName"] = (
    df["Model"].astype(str)
    .str.extract(r"\[(.*?)\]\(", expand=False)
    .fillna(df["Model"].astype(str))
)

num_cols = [
    "Active Parameters (B)",
    "Total Parameters (B)",
    "Mean (Task)",
    "Retrieval",
    "Reranking",
    "Pair Classification",
]

df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

ranked = df[
    (df["Active Parameters (B)"] <= 0.3)
    & df["Total Parameters (B)"].notna()
    & df[["Mean (Task)", "Retrieval", "Reranking", "Pair Classification"]].notna().all(axis=1)
].copy()

ranked["SoftSort"] = (
    0.60 * ranked["Retrieval"]
    + 0.25 * ranked["Reranking"]
    + 0.15 * ranked["Pair Classification"]
)

ranked = ranked.sort_values(["SoftSort", "Retrieval"], ascending=False).reset_index(drop=True)

cols = [
    "ModelName",
    "Active Parameters (B)",
    "Total Parameters (B)",
    "Mean (Task)",
    "Retrieval",
    "Reranking",
    "Pair Classification",
    "SoftSort",
]

ranked[cols].head(20)

,ModelName,Active Parameters (B),Total Parameters (B),Mean (Task),Retrieval,Reranking,Pair Classification,SoftSort
0,embeddinggemma-300m,0.106,0.308,74.00,72.73,73.67,77.71,73.7120
1,jina-embeddings-v5-text-nano,0.113,0.212,73.98,71.59,75.01,79.31,73.6030
2,harrier-oss-v1-270m,0.100,0.268,70.41,67.82,72.04,75.68,70.0540
3,granite-embedding-311m-multilingual-r2,0.110,0.312,69.38,68.32,71.47,70.11,69.3760
4,F2LLM-v2-330M,0.198,0.334,68.31,64.14,74.88,72.86,68.1330
5,granite-embedding-97m-multilingual-r2,0.028,0.097,67.02,65.60,69.66,68.17,67.0005
6,snowflake-arctic-embed-m-v2.0,0.113,0.305,66.93,64.08,72.82,68.63,66.9475
7,granite-embedding-278m-multilingual,0.086,0.278,66.51,63.69,70.85,69.69,66.3800
8,F2LLM-v2-160M,0.062,0.159,65.93,61.65,72.35,70.90,65.7125
9,multilingual-e5-base,0.086,0.278,65.22,60.30,72.51,71.04,64.9635


In [22]:
remove_reason = {
    "jina-embeddings-v5-text-nano": "non-commercial license",
    "snowflake-arctic-embed-m-v2.0": "runtime instability",
    "BidirLM-270M-Embedding": "runtime instability",
    "bilingual-embedding-base": "English-French only",
    "bilingual-embedding-small": "English-French only",
    "granite-embedding-278m-multilingual": "superseded by 311m-r2",
    "granite-embedding-107m-multilingual": "superseded by 97m-r2",
    "F2LLM-v2-80M": "weaker smaller variant",
    "paraphrase-multilingual-mpnet-base-v2": "general paraphrase model",
    "paraphrase-multilingual-MiniLM-L12-v2": "general paraphrase model",
    "Arabic-all-nli-triplet-Matryoshka": "language/domain mismatch",
    "granite-embedding-125m-english": "English-only",
    "bge-base-en-v1.5": "English-only",
}

top = ranked.head(20).copy()
top["RemoveReason"] = top["ModelName"].map(remove_reason)

final = top[top["RemoveReason"].isna()].copy()
removed = top[top["RemoveReason"].notna()].copy()

final[cols]

,ModelName,Active Parameters (B),Total Parameters (B),Mean (Task),Retrieval,Reranking,Pair Classification,SoftSort
0,embeddinggemma-300m,0.106,0.308,74.00,72.73,73.67,77.71,73.7120
2,harrier-oss-v1-270m,0.100,0.268,70.41,67.82,72.04,75.68,70.0540
3,granite-embedding-311m-multilingual-r2,0.110,0.312,69.38,68.32,71.47,70.11,69.3760
4,F2LLM-v2-330M,0.198,0.334,68.31,64.14,74.88,72.86,68.1330
5,granite-embedding-97m-multilingual-r2,0.028,0.097,67.02,65.60,69.66,68.17,67.0005
8,F2LLM-v2-160M,0.062,0.159,65.93,61.65,72.35,70.90,65.7125
9,multilingual-e5-base,0.086,0.278,65.22,60.30,72.51,71.04,64.9635


In [24]:
# removed[["ModelName", "RemoveReason", "Retrieval", "Reranking", "Pair Classification", "SoftSort"]]